# Gun Violence Tweet Emotion & Sentiment Pipeline

> See `README.md` for a short run guide.

End-to-end labeling for `sample_geotweets.csv`. Final emotion CSV keeps source metadata (`date`, `tags`, `place`, `geom`, `user_location`, `followers`, `friends`, `user_favorites`, `latitude`, `longitude`, `GPS`, `tweet_date`) plus labeled columns:

1. Loss / Grief Expression Percentage
2. Recent Shooting Event Mentions Percentage
3. Retaliation Language Percentage
4. Social Cohesion Breakdown Percentage
5. Trauma Exposure Percentage
6. Loss to Aggression Transition Signal Percentage
7. Final Emotion
8. Final Emotion Confidence Percentage
9. Emoticon Sentiment
10. Emoticon Sentiment Confidence Percentage
11. Tweet Summary (copied from the dedicated summaries folder)

**Tweet summaries (separate output):** while the same LLM is loaded for emotion labeling, a second pass writes one summary per tweet to `data/output/summaries/`.

**Emotion taxonomy:** fear · anger · grief/sadness · outrage · anxiety · helplessness · sympathy · community concern · resilience/hope · neutral/informational

**Models used**
1. **LLM:** Meta **Llama 3.2 3B Instruct** via Hugging Face Transformers (local, free; no Ollama). Auto-falls back to **Qwen2.5-3B-Instruct** if Llama access is gated.
2. **Validator:** CardiffNLP **Twitter-RoBERTa** emotion (pretrained)
3. **Emoticon:** lexicon-based emoji sentiment

**Final Emotion** = confidence-weighted ensemble of llama3.2 (weight 1.0) + Twitter-RoBERTa (weight 0.75).

Prerequisite: see `README.md` for packages. First LLM run downloads model weights (~2–6 GB).


## 0. Setup & configuration


In [ ]:
# Install / refresh core dependencies (safe to re-run)
import sys
import subprocess

for p in [
    "numpy",
    "pandas",
    "torch",
    "tqdm",
    "emoji",
    "safetensors",
    "accelerate",
    "transformers",
    "sentencepiece",
    "ipywidgets",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

print("Install cell finished.")


In [ ]:
from __future__ import annotations

import ast
import gc
import json
import os
import re
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_INPUT = ROOT / "data" / "input"
DATA_FILTERED = ROOT / "data" / "output" / "filtered"
DATA_LABELED = ROOT / "data" / "output" / "labeled"
DATA_SUMMARIES = ROOT / "data" / "output" / "summaries"
DATA_CHECKPOINTS = ROOT / "data" / "output" / "checkpoints"
for _d in (DATA_INPUT, DATA_FILTERED, DATA_LABELED, DATA_SUMMARIES, DATA_CHECKPOINTS):
    _d.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_INPUT / "sample_geotweets.csv"
OUT_DIR = DATA_LABELED
CHECKPOINT_DIR = DATA_CHECKPOINTS

FINAL_CSV = OUT_DIR / "tweets_with_emotion_labels_final.csv"
LLM_COMBINED_CSV = OUT_DIR / "tweets_llama_llm_labels.csv"
ROBERTA_CSV = OUT_DIR / "tweets_roberta_emotion_labels.csv"
EMOTICON_CSV = OUT_DIR / "tweets_emoticon_sentiment.csv"

# Dedicated summary outputs (created during the LLM labeling run)
SUMMARY_CSV = DATA_SUMMARIES / "tweet_summaries.csv"
SUMMARY_CKPT = DATA_SUMMARIES / "summary_resume_checkpoint.jsonl"

MAX_TWEETS: Optional[int] = None
CHECKPOINT_EVERY = 10
RESUME = True
MAX_TWEET_CHARS = 500
SUMMARY_MAX_NEW_TOKENS = 100

HF_TOKEN = os.environ.get("HF_TOKEN")

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("DATA_PATH:", DATA_PATH.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("DATA_SUMMARIES:", DATA_SUMMARIES.resolve())
print("CHECKPOINT_DIR:", CHECKPOINT_DIR.resolve())
print("DEVICE:", DEVICE)
print("HF_TOKEN set:", bool(HF_TOKEN))
print("MAX_TWEETS:", MAX_TWEETS)


In [ ]:
# Free local LLM via Hugging Face Transformers + Twitter-RoBERTa validator
# Preferred: same Llama 3.2 family as before (no Ollama). Fallback is ungated/open.
LLM_MODELS = {
    "llama32": {
        # Same model family previously used via Ollama
        "model_id": "meta-llama/Llama-3.2-3B-Instruct",
        # Fully open fallback if Llama weights are gated / HF token missing
        "fallback_model_id": "Qwen/Qwen2.5-3B-Instruct",
        "max_new_tokens": 220,
    },
}

ROBERTA_MODEL_ID = "cardiffnlp/twitter-roberta-base-emotion-multilabel-latest"

EMOTION_TAXONOMY = [
    "fear",
    "anger",
    "grief/sadness",
    "outrage",
    "anxiety",
    "helplessness",
    "sympathy",
    "community concern",
    "resilience/hope",
    "neutral/informational",
]

THEMATIC_SIGNAL_KEYS = [
    "loss_grief_expression_pct",
    "recent_shooting_event_mentions_pct",
    "retaliation_language_pct",
    "social_cohesion_breakdown_pct",
    "trauma_exposure_pct",
    "loss_to_aggression_transition_pct",
]

FINAL_COL_MAP = {
    "loss_grief_expression_pct": "Loss / Grief Expression Percentage",
    "recent_shooting_event_mentions_pct": "Recent Shooting Event Mentions Percentage",
    "retaliation_language_pct": "Retaliation Language Percentage",
    "social_cohesion_breakdown_pct": "Social Cohesion Breakdown Percentage",
    "trauma_exposure_pct": "Trauma Exposure Percentage",
    "loss_to_aggression_transition_pct": "Loss to Aggression Transition Signal Percentage",
    "final_emotion": "Final Emotion",
    "final_emotion_confidence_pct": "Final Emotion Confidence Percentage",
    "emoticon_sentiment": "Emoticon Sentiment",
    "emoticon_sentiment_confidence_pct": "Emoticon Sentiment Confidence Percentage",
    "tweet_summary": "Tweet Summary",
}

# Carry these columns from sample_geotweets.csv into the final export
SOURCE_EXPORT_COLS = [
    "date",
    "tags",
    "place",
    "geom",
    "user_location",
    "followers",
    "friends",
    "user_favorites",
    "latitude",
    "longitude",
    "GPS",
    "tweet_date",
]

LLM_WEIGHT = 1.0
ROBERTA_WEIGHT = 0.75

print("LLM preferred:", list(LLM_MODELS.values())[0]["model_id"])
print("LLM fallback :", list(LLM_MODELS.values())[0]["fallback_model_id"])
print("RoBERTa:", ROBERTA_MODEL_ID)
print("Emotion classes:", len(EMOTION_TAXONOMY))
print("Source export cols:", SOURCE_EXPORT_COLS)


## 1. Load & inspect source tweets

`sample_geotweets.csv` is **tab-separated** with multiline tweet text.


In [ ]:
assert DATA_PATH.exists(), f"Missing file: {DATA_PATH}"

raw = pd.read_csv(DATA_PATH, sep="\t", dtype=str, keep_default_na=False)
print("Raw shape:", raw.shape)
print("Columns:", list(raw.columns))
display(raw.head(3))


In [ ]:
REQUIRED_COLS = ["message_id", "date", "text"]
missing = [c for c in REQUIRED_COLS if c not in raw.columns]
assert not missing, f"Missing required columns: {missing}"

work = raw.copy()
work["text"] = work["text"].fillna("").astype(str).str.strip()
work = work[work["text"].str.len() > 0].copy()
work = work.drop_duplicates(subset=["message_id"], keep="first").reset_index(drop=True)

if MAX_TWEETS is not None:
    work = work.head(int(MAX_TWEETS)).copy()
    print(f"SMOKE TEST MODE: first {len(work)} tweets")

print("Working shape:", work.shape)
print("Unique message_id:", work["message_id"].nunique())
if "matched_incident" in work.columns:
    display(work["matched_incident"].value_counts().head())
print("\nSample tweet:\n", work.iloc[0]["text"][:400])


## 2. Helpers: emotion prompt, summary prompt, parsing, checkpoints

- **Emotion pass:** compact JSON for Final Emotion + thematic scores
- **Summary pass:** separate approved JSON prompt → `data/output/summaries/`


In [ ]:
_taxonomy_bullets = ", ".join(EMOTION_TAXONOMY)
SYSTEM_PROMPT = (
    "Classify a gun-violence-related tweet. Return ONLY compact JSON with keys:\n"
    "e = primary emotion (one of: " + _taxonomy_bullets + ")\n"
    "c = emotion confidence 0-100\n"
    "gv = gun_violence_related true/false\n"
    "lg = loss/grief expression 0-100\n"
    "rs = recent shooting event mention 0-100\n"
    "rl = retaliation language 0-100\n"
    "sc = social cohesion breakdown 0-100\n"
    "te = trauma exposure 0-100\n"
    "la = loss-to-aggression transition 0-100\n"
    'Example: {"e":"grief/sadness","c":85,"gv":true,"lg":70,"rs":90,"rl":5,"sc":40,"te":60,"la":10}'
)

# Approved dedicated summary prompt (separate LLM pass)
SUMMARY_SYSTEM_PROMPT = (
    "You summarize one social-media tweet in plain English.\n"
    "\n"
    "Rules:\n"
    '- Return ONLY valid JSON with a single key: "Tweet_Summary"\n'
    "- Tweet_Summary must be 1-2 short sentences\n"
    "- Describe what the tweet is saying / reacting to\n"
    "- Use only the tweet text; do not invent facts, names, places, or events that are not in the text\n"
    "- If the tweet is unclear, vague, or mostly links/emojis, say that briefly\n"
    "- Do not include hashtags, @mentions, or URLs unless they are needed to understand the meaning\n"
    "- Do not moralize, speculate about the author's intent beyond the text, or add analysis about gun violence unless the tweet itself does\n"
    "\n"
    "Example:\n"
    '{"Tweet_Summary":"The tweet mourns victims of a school shooting in Uvalde and asks for prayers."}'
)


def build_user_prompt(tweet_text: str, message_id: str) -> str:
    text = (tweet_text or "")[:MAX_TWEET_CHARS]
    return (
        f"message_id: {message_id}\n"
        f"tweet:\n\"\"\"\n{text}\n\"\"\"\n"
        "Respond with JSON only."
    )


def build_summary_user_prompt(tweet_text: str) -> str:
    text = (tweet_text or "")[:MAX_TWEET_CHARS]
    return (
        f'tweet:\n"""\n{text}\n"""\n'
        "Respond with JSON only."
    )


def extract_json_object(text: str) -> Optional[dict]:
    if not text:
        return None
    text = text.strip()
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.I).strip()
    text = re.sub(r"^```(?:json)?\s*|```\s*$", "", text, flags=re.I | re.M).strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return None
    candidate = m.group(0)
    for parser in (json.loads, ast.literal_eval):
        try:
            obj = parser(candidate)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return None


def clamp_pct(x: Any, default: float = 0.0) -> float:
    try:
        v = float(x)
    except Exception:
        return float(default)
    return float(max(0.0, min(100.0, v)))


def normalize_emotion(label: Any) -> str:
    if label is None:
        return "neutral/informational"
    s = str(label).strip().lower()
    aliases = {
        "grief": "grief/sadness",
        "sadness": "grief/sadness",
        "hope": "resilience/hope",
        "resilience": "resilience/hope",
        "neutral": "neutral/informational",
        "informational": "neutral/informational",
        "community_concern": "community concern",
    }
    s = aliases.get(s, s)
    if s in EMOTION_TAXONOMY:
        return s
    for e in EMOTION_TAXONOMY:
        if e in s or s in e:
            return e
    return "neutral/informational"


def clean_summary(text: Any) -> str:
    s = str(text or "").strip()
    s = re.sub(r"\s+", " ", s)
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1].strip()
    return s[:400]


def validate_llm_record(obj: Optional[dict]) -> dict:
    obj = obj or {}
    compact = {
        "primary_emotion": obj.get("primary_emotion", obj.get("e")),
        "emotion_confidence_pct": obj.get("emotion_confidence_pct", obj.get("c")),
        "gun_violence_related": obj.get("gun_violence_related", obj.get("gv", False)),
        "gun_violence_related_confidence_pct": obj.get(
            "gun_violence_related_confidence_pct", obj.get("gvc", 70)
        ),
        "brief_rationale": obj.get("brief_rationale", obj.get("r", "")),
        "loss_grief_expression_pct": obj.get("loss_grief_expression_pct", obj.get("lg")),
        "recent_shooting_event_mentions_pct": obj.get(
            "recent_shooting_event_mentions_pct", obj.get("rs")
        ),
        "retaliation_language_pct": obj.get("retaliation_language_pct", obj.get("rl")),
        "social_cohesion_breakdown_pct": obj.get(
            "social_cohesion_breakdown_pct", obj.get("sc")
        ),
        "trauma_exposure_pct": obj.get("trauma_exposure_pct", obj.get("te")),
        "loss_to_aggression_transition_pct": obj.get(
            "loss_to_aggression_transition_pct", obj.get("la")
        ),
        "secondary_emotions": obj.get("secondary_emotions", []),
    }
    secondary = compact["secondary_emotions"]
    if not isinstance(secondary, list):
        secondary = []
    secondary = [normalize_emotion(x) for x in secondary][:3]
    gv = compact["gun_violence_related"]
    if isinstance(gv, str):
        gv = gv.strip().lower() in {"1", "true", "yes", "y"}
    else:
        gv = bool(gv)
    out = {
        "primary_emotion": normalize_emotion(compact["primary_emotion"]),
        "emotion_confidence_pct": clamp_pct(compact["emotion_confidence_pct"], 0),
        "secondary_emotions": secondary,
        "gun_violence_related": gv,
        "gun_violence_related_confidence_pct": clamp_pct(
            compact["gun_violence_related_confidence_pct"], 70 if gv else 0
        ),
        "brief_rationale": str(compact["brief_rationale"] or "")[:240],
    }
    for k in THEMATIC_SIGNAL_KEYS:
        out[k] = clamp_pct(compact.get(k), 0)
    return out


def validate_summary_record(obj: Optional[dict], raw_text: str = "") -> str:
    obj = obj or {}
    summary = clean_summary(
        obj.get("Tweet_Summary", obj.get("tweet_summary", obj.get("s", obj.get("summary", ""))))
    )
    if not summary and raw_text:
        summary = clean_summary(raw_text.split("\n")[0])
    return summary


def ckpt_path(model_key: str) -> Path:
    return CHECKPOINT_DIR / "llama_resume_checkpoint.jsonl"


def load_checkpoint(model_key: str) -> Dict[str, dict]:
    """Load checkpoint; also accept legacy Ollama checkpoint name for resume."""
    candidates = [ckpt_path(model_key)]
    if model_key == "llama32":
        candidates.append(CHECKPOINT_DIR / "llm_ollama_llama32.jsonl")
    done: Dict[str, dict] = {}
    for path in candidates:
        if not path.exists():
            continue
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                    done[str(row["message_id"])] = row
                except Exception:
                    continue
        if done:
            print(f"Loaded {len(done)} checkpoint rows from {path.name}")
            break
    return done


def append_checkpoint(model_key: str, row: dict) -> None:
    with ckpt_path(model_key).open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_summary_checkpoint() -> Dict[str, dict]:
    done: Dict[str, dict] = {}
    if not SUMMARY_CKPT.exists():
        return done
    with SUMMARY_CKPT.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                mid = str(row.get("message_id", "")).strip()
                if mid and clean_summary(row.get("Tweet_Summary", "")):
                    done[mid] = row
            except Exception:
                continue
    if done:
        print(f"Loaded {len(done)} summary checkpoint rows from {SUMMARY_CKPT.name}")
    return done


def append_summary_checkpoint(row: dict) -> None:
    with SUMMARY_CKPT.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


# Parser self-tests
_demo = '{"e":"grief/sadness","c":88,"gv":true,"lg":90,"rs":80,"rl":5,"sc":40,"te":70,"la":10}'
parsed = validate_llm_record(extract_json_object(_demo))
assert parsed["primary_emotion"] == "grief/sadness"
_sum_demo = '{"Tweet_Summary":"The tweet expresses mourning after a school shooting."}'
assert "mourning" in validate_summary_record(extract_json_object(_sum_demo)).lower()
print("Parser self-test OK:", parsed)
print("Emotion prompt length:", len(SYSTEM_PROMPT))
print("Summary prompt length:", len(SUMMARY_SYSTEM_PROMPT))


## 3. Emoticon / emoji sentiment

Independent lexicon scoring → `Emoticon Sentiment` + confidence %.


In [ ]:
EMOJI_SENTIMENT = {
    "❤️": 1.0, "❤": 1.0, "💕": 0.9, "💗": 0.9, "💙": 0.8, "💜": 0.8, "💛": 0.8,
    "🙏": 0.8, "🙏🏻": 0.8, "🙏🏼": 0.8, "🙏🏽": 0.8, "🙏🏾": 0.8, "🙏🏿": 0.8,
    "🕊️": 0.7, "🕊": 0.7, "🌹": 0.6, "🌸": 0.5, "✨": 0.4, "💪": 0.5,
    "😊": 0.7, "🙂": 0.4, "😄": 0.7, "🤗": 0.6, "😇": 0.6, "👍": 0.5,
    "😢": -0.7, "😭": -0.8, "😞": -0.5, "☹️": -0.5, "😩": -0.6, "💔": -0.9,
    "😔": -0.5, "🖤": -0.3,
    "😱": -0.8, "😨": -0.7, "😰": -0.7, "😟": -0.5, "🫨": -0.6,
    "😡": -0.9, "🤬": -1.0, "😠": -0.8, "💢": -0.7, "👿": -0.8, "🖕": -0.9,
    "😮": -0.2, "‼️": -0.2, "❗": -0.1, "❓": 0.0, "⚠️": -0.3, "🚨": -0.4,
    "🔫": -0.6, "💀": -0.7, "☠️": -0.7, "🩸": -0.8,
    "🔥": 0.1, "👏": 0.4, "🎉": 0.6, "🤢": -0.6, "🤮": -0.7,
}

EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE,
)


def extract_emojis(text: str) -> List[str]:
    return EMOJI_RE.findall(text or "")


def score_emoticon_sentiment(text: str) -> dict:
    emojis = extract_emojis(text)
    if not emojis:
        return {
            "emojis": [],
            "emoji_count": 0,
            "emoticon_sentiment": "none",
            "emoticon_sentiment_confidence_pct": 0.0,
            "emoticon_raw_score": 0.0,
        }
    scores = []
    for e in emojis:
        if e in EMOJI_SENTIMENT:
            scores.append(EMOJI_SENTIMENT[e])
        else:
            hit = next((v for k, v in EMOJI_SENTIMENT.items() if k in e or e in k), 0.0)
            scores.append(hit)
    raw = float(np.mean(scores)) if scores else 0.0
    n_informative = sum(1 for s in scores if abs(s) >= 0.2)
    conf = min(100.0, (0.35 + 0.15 * n_informative) * abs(raw) * 100 + 10 * min(len(emojis), 3))
    conf = float(max(0.0, min(100.0, conf)))
    if raw >= 0.25:
        label = "positive"
    elif raw <= -0.25:
        label = "negative"
    else:
        label = "neutral"
        conf = min(conf, 55.0)
    return {
        "emojis": emojis,
        "emoji_count": len(emojis),
        "emoticon_sentiment": label,
        "emoticon_sentiment_confidence_pct": round(conf, 2),
        "emoticon_raw_score": round(raw, 4),
    }


emo_rows = []
for _, r in work.iterrows():
    s = score_emoticon_sentiment(r["text"])
    emo_rows.append(
        {
            "message_id": r["message_id"],
            **s,
            "emojis": json.dumps(s["emojis"], ensure_ascii=False),
        }
    )

emoticon_df = pd.DataFrame(emo_rows)
emoticon_df.to_csv(EMOTICON_CSV, index=False)
print("Emoticon rows:", len(emoticon_df))
display(emoticon_df["emoticon_sentiment"].value_counts())
print("Tweets with emojis:", int((emoticon_df["emoji_count"] > 0).sum()))
display(emoticon_df[emoticon_df["emoji_count"] > 0].head(5))
print("Saved:", EMOTICON_CSV.resolve())


## 4. LLM labeling + tweet summaries (`Llama-3.2-3B-Instruct`)

Runs the same Llama 3.2 instruct model **locally via Transformers** (no Ollama server).

1. **Emotion / theme pass** → checkpoints under `data/output/checkpoints/`
2. **Summary pass** (approved summary prompt, same loaded model) → `data/output/summaries/`

- Preferred model: `meta-llama/Llama-3.2-3B-Instruct`
- Automatic fallback: `Qwen/Qwen2.5-3B-Instruct`
- Both passes resume from their own checkpoints


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

_llm_tokenizer = None
_llm_model = None
_llm_loaded_id: Optional[str] = None


def _hf_kwargs() -> dict:
    return {"token": HF_TOKEN} if HF_TOKEN else {}


def _torch_dtype():
    if DEVICE.type in {"cuda", "mps"}:
        return torch.float16
    return torch.float32


def load_causal_lm(model_id: str):
    """Load a chat-instruct causal LM onto DEVICE."""
    global _llm_tokenizer, _llm_model, _llm_loaded_id
    if _llm_loaded_id == model_id and _llm_model is not None:
        return _llm_tokenizer, _llm_model

    # Free previous model if switching
    if _llm_model is not None:
        try:
            del _llm_model
            del _llm_tokenizer
        except Exception:
            pass
        gc.collect()
        if DEVICE.type == "mps":
            try:
                torch.mps.empty_cache()
            except Exception:
                pass

    print(f"Loading LLM weights: {model_id} on {DEVICE} ...")
    tok = AutoTokenizer.from_pretrained(model_id, **_hf_kwargs())
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Some local/cached tokenizers ship without chat_template — set a Llama-3 style default.
    if not getattr(tok, "chat_template", None):
        tok.chat_template = (
            "{% for message in messages %}"
            "{% if message['role'] == 'system' %}"
            "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\n' + message['content'] + '<|eot_id|>' }}"
            "{% elif message['role'] == 'user' %}"
            "{{ '<|start_header_id|>user<|end_header_id|>\\n\\n' + message['content'] + '<|eot_id|>' }}"
            "{% elif message['role'] == 'assistant' %}"
            "{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' + message['content'] + '<|eot_id|>' }}"
            "{% endif %}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}"
            "{% endif %}"
        )
        print("Tokenizer had no chat_template; applied Llama-3 style fallback template.")

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=_torch_dtype(),
        low_cpu_mem_usage=True,
        **_hf_kwargs(),
    )
    model.to(DEVICE)
    model.eval()
    _llm_tokenizer, _llm_model, _llm_loaded_id = tok, model, model_id
    print("LLM ready:", model_id)
    return tok, model


def resolve_llm_model_id(cfg: dict) -> str:
    """Prefer Llama 3.2; fall back to ungated open model if download/auth fails."""
    preferred = cfg["model_id"]
    fallback = cfg.get("fallback_model_id") or preferred
    try:
        # Lightweight probe: fetch tokenizer config only
        AutoTokenizer.from_pretrained(preferred, **_hf_kwargs())
        return preferred
    except Exception as e:
        print(f"Could not access {preferred}: {e}")
        print(f"Falling back to open model: {fallback}")
        return fallback


def format_chat_prompt(tok, messages: List[dict]) -> str:
    """Build a chat prompt; never fail just because chat_template is missing."""
    if getattr(tok, "chat_template", None):
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception as e:
            print(f"apply_chat_template failed ({e}); using manual prompt format.")

    # Manual fallback (works for Llama-3 / most instruct models)
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        parts.append(f"<|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>")
    parts.append("<|start_header_id|>assistant<|end_header_id|>\n\n")
    return "".join(parts)


def generate_json_for_tweet(model_cfg: dict, message_id: str, tweet_text: str) -> Tuple[dict, str]:
    tok, model = load_causal_lm(model_cfg["_resolved_model_id"])
    user = build_user_prompt(tweet_text, message_id)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    prompt = format_chat_prompt(tok, messages)
    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    max_new = int(model_cfg.get("max_new_tokens", 160))

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

    gen_ids = output_ids[0][inputs["input_ids"].shape[-1] :]
    raw_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    parsed = validate_llm_record(extract_json_object(raw_text))
    return parsed, raw_text


def run_llm_model_on_dataframe(model_key: str, df: pd.DataFrame) -> pd.DataFrame:
    cfg = dict(LLM_MODELS[model_key])
    print("=" * 72)
    print(f"Running LLM: {model_key}")
    print("=" * 72)

    done = load_checkpoint(model_key) if RESUME else {}
    # Re-try rows that were checkpointed with an error
    for mid, row in list(done.items()):
        if str(row.get("error") or "").strip():
            done.pop(mid, None)
    print(f"Already checkpointed: {len(done)} / {len(df)}")
    pending = df[~df["message_id"].astype(str).isin(done.keys())].copy()

    if pending.empty:
        print("Nothing pending - using checkpoint only.")
        # Prefer model_id stored in checkpoint rows when available
        model_id = None
        if done:
            model_id = next(iter(done.values())).get("model_id")
        model_id = model_id or cfg["model_id"]
    else:
        model_id = resolve_llm_model_id(cfg)
        cfg["_resolved_model_id"] = model_id
        load_causal_lm(model_id)
        n_since = 0
        for _, row in tqdm(pending.iterrows(), total=len(pending), desc=model_key):
            mid = str(row["message_id"])
            try:
                parsed, raw_text = generate_json_for_tweet(cfg, mid, row["text"])
                err = ""
            except Exception as e:
                parsed = validate_llm_record(None)
                raw_text = ""
                err = repr(e)
                print(f"ERROR {mid}: {err}")
            rec = {
                "message_id": mid,
                "model_key": model_key,
                "model_id": model_id,
                **parsed,
                "secondary_emotions": json.dumps(parsed["secondary_emotions"], ensure_ascii=False),
                "raw_response": raw_text[:4000],
                "error": err,
            }
            append_checkpoint(model_key, rec)
            done[mid] = rec
            n_since += 1
            if n_since % CHECKPOINT_EVERY == 0:
                print(f"  progress: {len(done)} / {len(df)}")

    rows = [done[str(m)] for m in df["message_id"].astype(str).tolist() if str(m) in done]
    out = pd.DataFrame(rows)
    out_path = CHECKPOINT_DIR / "llama_resume_checkpoint_table.csv"
    out.to_csv(out_path, index=False)
    print(f"Saved table: {out_path} | rows={len(out)}")
    if len(out):
        display(out["primary_emotion"].value_counts())
        display(out[THEMATIC_SIGNAL_KEYS].astype(float).mean().round(2))
        display(out.head(3))
    return out




def generate_summary_for_tweet(model_cfg: dict, tweet_text: str) -> Tuple[str, str]:
    """Dedicated summary call using the approved SUMMARY_SYSTEM_PROMPT."""
    tok, model = load_causal_lm(model_cfg["_resolved_model_id"])
    user = build_summary_user_prompt(tweet_text)
    messages = [
        {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    prompt = format_chat_prompt(tok, messages)
    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(SUMMARY_MAX_NEW_TOKENS),
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

    gen_ids = output_ids[0][inputs["input_ids"].shape[-1] :]
    raw_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    summary = validate_summary_record(extract_json_object(raw_text), raw_text)
    return summary, raw_text


def run_summaries_on_dataframe(model_key: str, df: pd.DataFrame) -> pd.DataFrame:
    """Second LLM pass: write structured summaries to data/output/summaries/."""
    cfg = dict(LLM_MODELS[model_key])
    print("=" * 72)
    print(f"Running tweet summaries: {model_key}")
    print("=" * 72)

    done = load_summary_checkpoint() if RESUME else {}
    print(f"Already summarized: {len(done)} / {len(df)}")
    pending = df[~df["message_id"].astype(str).isin(done.keys())].copy()

    if pending.empty:
        print("Nothing pending - using summary checkpoint only.")
        model_id = None
        if done:
            model_id = next(iter(done.values())).get("model_id")
        model_id = model_id or cfg.get("fallback_model_id") or cfg["model_id"]
    else:
        # Prefer the already-loaded emotion model if present
        model_id = _llm_loaded_id or resolve_llm_model_id(cfg)
        cfg["_resolved_model_id"] = model_id
        load_causal_lm(model_id)
        n_since = 0
        for _, row in tqdm(pending.iterrows(), total=len(pending), desc="summaries"):
            mid = str(row["message_id"])
            try:
                summary, raw_text = generate_summary_for_tweet(cfg, row["text"])
                err = ""
            except Exception as e:
                summary = ""
                raw_text = ""
                err = repr(e)
                print(f"SUMMARY ERROR {mid}: {err}")
            rec = {
                "message_id": mid,
                "model_key": model_key,
                "model_id": model_id,
                "Tweet_Summary": summary,
                "raw_response": raw_text[:2000],
                "error": err,
            }
            append_summary_checkpoint(rec)
            done[mid] = rec
            n_since += 1
            if n_since % CHECKPOINT_EVERY == 0:
                print(f"  summary progress: {len(done)} / {len(df)}")

    # Structured export: tweet page fields + Tweet_Summary
    meta_cols = [
        c
        for c in [
            "message_id",
            "tweet_date",
            "date",
            "place",
            "user_location",
            "latitude",
            "longitude",
            "GPS",
            "matched_incident",
            "text",
            "tags",
            "retweets",
            "tweet_favorites",
            "followers",
            "friends",
            "user_favorites",
            "user_id",
            "user_name",
        ]
        if c in df.columns
    ]
    out = df[meta_cols].copy()
    out["message_id"] = out["message_id"].astype(str)
    out["Tweet_Summary"] = out["message_id"].map(
        lambda mid: str((done.get(mid) or {}).get("Tweet_Summary", "") or "")
    )
    out["text"] = (
        out["text"].astype(str).str.replace("\r\n", " ", regex=False).str.replace("\n", " ", regex=False)
    )
    out["Tweet_Summary"] = (
        out["Tweet_Summary"]
        .astype(str)
        .str.replace("\r\n", " ", regex=False)
        .str.replace("\n", " ", regex=False)
    )
    # Excel-friendly: keep message_id as text + UTF-8 BOM
    csv_out = out.copy()
    csv_out["message_id"] = "'" + csv_out["message_id"].astype(str)
    csv_out.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

    n_ok = int((out["Tweet_Summary"].astype(str).str.len() > 5).sum())
    print(f"Saved summaries: {SUMMARY_CSV.resolve()} | rows={len(out)} | filled={n_ok}")
    display(out[["message_id", "place", "Tweet_Summary"]].head(5))
    return out


print("LLM helper ready (Hugging Face Transformers, no Ollama; summaries enabled).")


In [ ]:
# Run Llama 3.2 (HF): emotion pass, then dedicated summary pass (same model)
llm_tables = {}
for key in LLM_MODELS:
    llm_tables[key] = run_llm_model_on_dataframe(key, work)

for k, d in llm_tables.items():
    n_err = int((d["error"].astype(str).str.len() > 0).sum()) if "error" in d.columns else -1
    print(f"{k}: {len(d)} rows | errors={n_err}")

# Summaries → data/output/summaries/ (approved SUMMARY_SYSTEM_PROMPT)
summary_key = next(iter(LLM_MODELS))
summary_df = run_summaries_on_dataframe(summary_key, work)
print("Summary rows:", len(summary_df))


In [ ]:
base_cols = ["message_id", "text"]
for c in SOURCE_EXPORT_COLS:
    if c in work.columns and c not in base_cols:
        base_cols.append(c)
combined = work[base_cols].copy()

for key, d in llm_tables.items():
    cols = [
        "primary_emotion",
        "emotion_confidence_pct",
        "secondary_emotions",
        "gun_violence_related",
        "gun_violence_related_confidence_pct",
        *THEMATIC_SIGNAL_KEYS,
        "brief_rationale",
        "error",
    ]
    keep = ["message_id"] + [c for c in cols if c in d.columns]
    sub = d[keep].copy()
    sub = sub.rename(columns={c: f"{key}__{c}" for c in sub.columns if c != "message_id"})
    combined = combined.merge(sub, on="message_id", how="left")

combined.to_csv(LLM_COMBINED_CSV, index=False)
print("Combined LLM shape:", combined.shape)
print("Saved:", LLM_COMBINED_CSV.resolve())
display(combined.head(2))


## 5. Twitter-RoBERTa emotion validation (pretrained)

CardiffNLP multilabel tweet emotion model. Labels are mapped onto our taxonomy.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer as HFTokenizer

ROBERTA_TO_TAXONOMY = {
    "anger": {"anger": 1.0, "outrage": 0.55},
    "anticipation": {"anxiety": 0.45, "community concern": 0.25},
    "disgust": {"outrage": 0.8, "anger": 0.4},
    "fear": {"fear": 1.0, "anxiety": 0.5},
    "joy": {"resilience/hope": 0.7, "sympathy": 0.2},
    "love": {"sympathy": 0.8, "resilience/hope": 0.3},
    "optimism": {"resilience/hope": 1.0},
    "pessimism": {"helplessness": 0.8, "grief/sadness": 0.35},
    "sadness": {"grief/sadness": 1.0, "helplessness": 0.35},
    "surprise": {"anxiety": 0.35, "neutral/informational": 0.2},
    "trust": {"community concern": 0.5, "sympathy": 0.45, "resilience/hope": 0.25},
}


def free_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if DEVICE.type == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


if RESUME and ROBERTA_CSV.exists():
    roberta_df = pd.read_csv(ROBERTA_CSV, dtype={"message_id": str})
    roberta_df = (
        work[["message_id"]]
        .astype(str)
        .merge(roberta_df, on="message_id", how="left")
    )
    print(f"Loaded RoBERTa labels from {ROBERTA_CSV} | rows={len(roberta_df)}")
else:
    kw = {}
    if HF_TOKEN:
        kw["token"] = HF_TOKEN

    tokenizer_r = HFTokenizer.from_pretrained(ROBERTA_MODEL_ID, **kw)
    model_r = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL_ID, **kw)
    model_r.to(DEVICE)
    model_r.eval()
    id2label_r = model_r.config.id2label
    print("RoBERTa labels:", [id2label_r[i] for i in range(len(id2label_r))])

    texts = work["text"].tolist()
    results = []
    batch_size = 16
    for i in tqdm(range(0, len(texts), batch_size), desc="roberta"):
        batch = texts[i : i + batch_size]
        enc = tokenizer_r(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = model_r(**enc).logits
            probs = torch.sigmoid(logits).detach().cpu().numpy()
        for row_prob in probs:
            label_scores = {str(id2label_r[j]).lower(): float(row_prob[j]) for j in range(len(row_prob))}
            tax = {e: 0.0 for e in EMOTION_TAXONOMY}
            for lab, p in label_scores.items():
                mapping = ROBERTA_TO_TAXONOMY.get(lab, {})
                if not mapping:
                    tax["neutral/informational"] += 0.05 * p
                for t, w in mapping.items():
                    tax[t] += p * w
            mass = sum(tax.values())
            if mass <= 1e-9:
                top = "neutral/informational"
                conf = 0.0
            else:
                tax = {k: v / mass for k, v in tax.items()}
                top = max(tax, key=tax.get)
                conf = 100.0 * tax[top]
            results.append(
                {
                    "roberta_raw_scores": json.dumps(label_scores),
                    "roberta_taxonomy_scores": json.dumps(tax),
                    "roberta_primary_emotion": top,
                    "roberta_emotion_confidence_pct": round(float(conf), 2),
                }
            )

    free_model(model_r, tokenizer_r)
    roberta_df = pd.DataFrame(results)
    roberta_df.insert(0, "message_id", work["message_id"].values)
    roberta_df.to_csv(ROBERTA_CSV, index=False)
    print("Saved:", ROBERTA_CSV.resolve())

print("RoBERTa rows:", len(roberta_df))
display(roberta_df["roberta_primary_emotion"].value_counts())
display(roberta_df.head(5))


## 6. Ensemble Final Emotion + thematic percentages

- **Final Emotion:** confidence-weighted vote (llama3.2 × 1.0 + RoBERTa × 0.75)
- **Thematic % columns:** from llama3.2 scores
- **Emoticon columns:** from Section 3


In [ ]:
def ensemble_row(row: pd.Series) -> Tuple[str, float]:
    votes = {e: 0.0 for e in EMOTION_TAXONOMY}
    for key in LLM_MODELS:
        emo = row.get(f"{key}__primary_emotion")
        conf = row.get(f"{key}__emotion_confidence_pct")
        if pd.isna(emo):
            continue
        emo = normalize_emotion(emo)
        try:
            conf_f = float(conf)
        except Exception:
            conf_f = 50.0
        votes[emo] += LLM_WEIGHT * (conf_f / 100.0)

    remo = row.get("roberta_primary_emotion")
    rconf = row.get("roberta_emotion_confidence_pct")
    if not pd.isna(remo):
        remo = normalize_emotion(remo)
        try:
            rconf_f = float(rconf)
        except Exception:
            rconf_f = 50.0
        votes[remo] += ROBERTA_WEIGHT * (rconf_f / 100.0)

    total = sum(votes.values())
    if total <= 1e-9:
        return "neutral/informational", 0.0
    winner = max(votes, key=votes.get)
    return winner, round(100.0 * votes[winner] / total, 2)


ens = combined.merge(
    roberta_df[
        [
            "message_id",
            "roberta_primary_emotion",
            "roberta_emotion_confidence_pct",
            "roberta_taxonomy_scores",
        ]
    ],
    on="message_id",
    how="left",
)
ens = ens.merge(
    emoticon_df[
        [
            "message_id",
            "emoticon_sentiment",
            "emoticon_sentiment_confidence_pct",
            "emoji_count",
            "emojis",
        ]
    ],
    on="message_id",
    how="left",
)

finals = [ensemble_row(row) for _, row in ens.iterrows()]
ens["final_emotion"] = [x[0] for x in finals]
ens["final_emotion_confidence_pct"] = [x[1] for x in finals]

for k in THEMATIC_SIGNAL_KEYS:
    cols = [f"{m}__{k}" for m in LLM_MODELS if f"{m}__{k}" in ens.columns]
    ens[k] = ens[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1).round(2)

gv_cols = [
    f"{m}__gun_violence_related"
    for m in LLM_MODELS
    if f"{m}__gun_violence_related" in ens.columns
]


def _as_bool(x):
    if isinstance(x, str):
        return x.strip().lower() in {"1", "true", "yes", "y"}
    return bool(x)


if gv_cols:
    ens["gun_violence_related_majority"] = ens[gv_cols].apply(
        lambda r: sum(_as_bool(v) for v in r) >= (len(gv_cols) / 2), axis=1
    )

print("Final emotion distribution:")
display(ens["final_emotion"].value_counts())
print("\nMean thematic signals:")
display(ens[THEMATIC_SIGNAL_KEYS].mean().round(2))

for key in LLM_MODELS:
    col = f"{key}__primary_emotion"
    if col in ens.columns:
        rate = (ens[col].map(normalize_emotion) == ens["final_emotion"]).mean()
        print(f"{key} matches Final Emotion: {rate:.1%}")
rate_r = (ens["roberta_primary_emotion"].map(normalize_emotion) == ens["final_emotion"]).mean()
print(f"roberta matches Final Emotion: {rate_r:.1%}")
display(ens.head(3))


# Attach dedicated LLM summaries (from data/output/summaries/)
if "summary_df" in globals() and summary_df is not None and "Tweet_Summary" in summary_df.columns:
    ens = ens.merge(
        summary_df[["message_id", "Tweet_Summary"]].rename(columns={"Tweet_Summary": "tweet_summary"}),
        on="message_id",
        how="left",
    )
    ens["tweet_summary"] = ens["tweet_summary"].fillna("").astype(str)
else:
    ens["tweet_summary"] = ""


## 7. Export final emotion CSV (+ summaries already in `data/output/summaries/`)


In [ ]:
export_cols_internal = []
for c in SOURCE_EXPORT_COLS:
    if c not in export_cols_internal:
        export_cols_internal.append(c)
if "gun_violence_related_majority" in ens.columns:
    export_cols_internal.append("gun_violence_related_majority")

export_cols_internal += THEMATIC_SIGNAL_KEYS + [
    "tweet_summary",
    "final_emotion",
    "final_emotion_confidence_pct",
    "emoticon_sentiment",
    "emoticon_sentiment_confidence_pct",
    "emoji_count",
    "emojis",
    "roberta_primary_emotion",
    "roberta_emotion_confidence_pct",
]
for key in LLM_MODELS:
    export_cols_internal += [
        f"{key}__primary_emotion",
        f"{key}__emotion_confidence_pct",
    ]

available = [c for c in export_cols_internal if c in ens.columns]
final_df = ens[available].copy()
final_df = final_df.rename(
    columns={k: v for k, v in FINAL_COL_MAP.items() if k in final_df.columns}
)
final_df.to_csv(FINAL_CSV, index=False)

print("Final CSV:", FINAL_CSV.resolve())
print("Shape:", final_df.shape)
print("Columns:")
for c in final_df.columns:
    print(" -", c)
display(final_df.head(5))


## 8. End-to-end QC checklist


In [ ]:
required_final = list(FINAL_COL_MAP.values())
missing_final = [c for c in required_final if c not in final_df.columns]
assert not missing_final, f"Missing final columns: {missing_final}"
missing_source = [c for c in SOURCE_EXPORT_COLS if c not in final_df.columns]
assert not missing_source, f"Missing source columns in final output: {missing_source}"
assert len(final_df) == len(work), "Row count mismatch vs working set"
assert "message_id" not in final_df.columns and "text" not in final_df.columns
assert SUMMARY_CSV.exists(), f"Missing summaries file: {SUMMARY_CSV}"

print("QC PASS")
print(f"  rows              : {len(final_df)}")
print(f"  required columns  : {len(required_final)} present")
print(f"  source columns    : {len(SOURCE_EXPORT_COLS)} present")
print(f"  LLM model         : Llama-3.2-3B-Instruct via Hugging Face (local)")
print(f"  validator         : Twitter-RoBERTa")
print(f"  emotion output    : {FINAL_CSV.resolve()}")
print(f"  summaries output  : {SUMMARY_CSV.resolve()}")
print("\nTop emotions:")
display(final_df["Final Emotion"].value_counts())
print("\nPreview of source + required columns:")
display(final_df[[*SOURCE_EXPORT_COLS, *required_final]].head(10))
